# 14 — Demographics theo persona P1–P6
Giữ nguyên assignment của notebook 10. Ghép thông tin đúng `customer_id + tháng`, mô tả 7 biến và kiểm tra khác biệt theo tháng. **Không refit cây, không dự đoán acquisition.**

## Cách chạy trên Kaggle
1. Add Input: dữ liệu gốc Santander và output notebook 10 có `personas_tree_YYYY-MM.parquet`.
2. Kiểm tra `RAW_PATH`, `PERSONA_DIR`, `MONTHS` trong cell cấu hình. Nếu tự tìm thấy nhiều nguồn, đặt đường dẫn rõ ràng.
3. Restart & Run All. Đầu ra nằm trong `persona_demographics_outputs`.

Notebook này chưa chạy trên dữ liệu Santander trong phiên tạo file. Kiểm tra cục bộ dùng dữ liệu giả lập, không phải kết quả thực nghiệm.

**Mẫu số:** baseline của mỗi tháng là toàn bộ khách được gán P1–P6 trong đúng file assignment; tỷ trọng categories tính trên toàn nhóm, gồm Missing. Thống kê numeric tính trên phần hợp lệ, luôn kèm coverage. Mỗi khách chỉ một dòng mỗi tháng. Không impute để viết chân dung, không gộp 17 tháng thành một snapshot.

In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
from IPython.display import display
RAW_PATH = None  # CSV hoặc ZIP chứa train_ver2.csv
PERSONA_DIR = None  # folder output 10; không phải file ipynb
MONTHS = ['2015-12','2016-01','2016-02','2016-03','2016-04','2016-05']
OUT = Path('/kaggle/working/persona_demographics_outputs' if Path('/kaggle').exists() else './persona_demographics_outputs')
OUT.mkdir(parents=True, exist_ok=True)
SEARCH = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
CHUNKSIZE = 400_000
# Ngưỡng dưới đây là tiêu chí mô tả ứng viên, không phải kiểm định ý nghĩa thống kê.
MIN_GROUP_N = 1000
MIN_VALID_SHARE = .70
MIN_ABS_SHARE_DIFF = .05  # 5 điểm phần trăm
MIN_CATEGORY_N = 100
AGE_MAX = 110  # ngưỡng screening cấu hình được; tuổi >110 là nghi vấn, không khẳng định sai
MIN_STABLE_MONTHS = 3
MIN_STABLE_FRACTION = .80
MISSING = '__MISSING__'
PIDS = ['P1','P2','P3','P4','P5','P6']
RAW_COLS = ['fecha_dato','ncodpers','age','renta','antiguedad','canal_entrada','segmento','sexo','nomprov']
NUMERIC = ['age','renta','antiguedad']
CATEGORICAL = ['age_group','tenure_group','canal_entrada','segmento','sexo','nomprov']
assert MONTHS == sorted(set(MONTHS)) and all(str(pd.Period(m,'M')) == m for m in MONTHS)

def resolve_inputs():
    raw = Path(RAW_PATH) if RAW_PATH else None
    if raw is None:
        hits = sorted(SEARCH.rglob('train_ver2.csv.zip'))
        if not hits: hits = sorted(SEARCH.rglob('train_ver2.csv'))
        if len(hits) != 1: raise ValueError(f'Đặt RAW_PATH; tìm thấy {len(hits)} nguồn: {hits}')
        raw = hits[0]
    root = Path(PERSONA_DIR) if PERSONA_DIR else None
    if root is None:
        roots = sorted({p.parent for p in SEARCH.rglob('personas_tree_????-??.parquet')})
        if len(roots) != 1: raise ValueError(f'Đặt PERSONA_DIR; tìm thấy {roots}')
        root = roots[0]
    if not raw.is_file(): raise FileNotFoundError(raw)
    for m in MONTHS:
        if not (root/f'personas_tree_{m}.parquet').is_file():
            raise FileNotFoundError(f'Thiếu assignment {m}; xuất từ 10 hoặc sửa MONTHS. Không bỏ qua âm thầm.')
    return raw, root

RAW, PERSONAS = resolve_inputs()
print('Raw:', RAW, '\nPersona:', PERSONAS, '\nMonths:', MONTHS)

## 1. Đọc raw theo chunk và giữ 7 biến cần thiết
Không đọc 24 cờ sản phẩm, không tải toàn bộ 48 cột. Tuổi dưới 18 được giữ nguyên; khoảng tuổi chỉ là cách trình bày. Thu nhập <0, thâm niên <0 và tuổi ngoài [0, AGE_MAX] được giữ ở cột `_raw`, đánh dấu nghi vấn rồi loại khỏi thống kê numeric. Không suy ra nghề nghiệp, gia đình hay mức độ trung thành từ demographics.

In [ ]:
def read_raw_months(path, months):
    pieces = {m: [] for m in months}
    def consume(source):
        reader = pd.read_csv(source, usecols=RAW_COLS, dtype='string', chunksize=CHUNKSIZE,
                             na_values=['NA',' NA',''], keep_default_na=True)
        for chunk in reader:
            dates = pd.to_datetime(chunk.fecha_dato, errors='coerce')
            if dates.isna().any(): raise ValueError('fecha_dato thiếu/không hợp lệ')
            labels = dates.dt.strftime('%Y-%m')
            for m in months:
                q = chunk.loc[labels.eq(m)].copy()
                if len(q):
                    q['as_of_month'] = m
                    pieces[m].append(q)
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if Path(n).name == 'train_ver2.csv']
            if len(names) != 1: raise ValueError('ZIP phải có đúng một train_ver2.csv')
            with z.open(names[0]) as f: consume(f)
    else: consume(path)
    result = {}
    for m, parts in pieces.items():
        if not parts: raise ValueError(f'Raw thiếu tháng {m}')
        q = pd.concat(parts, ignore_index=True)
        ids = pd.to_numeric(q.ncodpers, errors='raise')
        assert ids.notna().all() and ids.mod(1).eq(0).all()
        q['customer_id'] = ids.astype('int64')
        assert not q.customer_id.duplicated().any(), f'{m}: trùng snapshot khách'
        result[m] = clean_demographics(q)
    return result

def clean_demographics(q):
    q = q.copy()
    for c in RAW_COLS[2:]:
        s = q[c].astype('string').str.strip().replace('', pd.NA)
        q[c+'_missing_raw'] = s.isna()
        if c in NUMERIC:
            v = pd.to_numeric(s, errors='coerce').astype(float)
            bad = s.notna() & v.isna()
            bad |= v.notna() & ~np.isfinite(v)
            bad |= v.lt(0)
            if c == 'age': bad |= v.gt(AGE_MAX)
            q[c+'_raw'] = s
            q[c+'_invalid'] = bad
            q[c] = v.mask(bad)
        else:
            q[c] = s.fillna(MISSING)
    q['age_group'] = pd.cut(q.age, [-1,17,25,35,45,55,65,np.inf],
                           labels=['0–17','18–25','26–35','36–45','46–55','56–65','66+']).astype('string').fillna(MISSING)
    q['tenure_group'] = pd.cut(q.antiguedad, [-1,6,12,36,60,120,np.inf],
                              labels=['0–6m','7–12m','13–36m','37–60m','61–120m','121m+']).astype('string').fillna(MISSING)
    return q

raw_months = read_raw_months(RAW, MONTHS)
print('Raw tháng đã đọc:', {m: len(q) for m,q in raw_months.items()})

## 2. Ghép đúng tháng, kiểm tra coverage
Khách không có assignment P1–P6 không được tự gán persona. Chỉ lấy cột định danh/persona từ output 10 để tránh xung đột feature cũ. Báo riêng tỷ lệ raw snapshot được gán persona; baseline không đại diện toàn known population hoặc khách đang vắng snapshot.

In [ ]:
def join_personas(raw, assignments, month):
    p = assignments.copy()
    assert p.customer_id.notna().all() and p.customer_id.is_unique
    for col in ['as_of_month','month']:
        if col in p:
            assert p[col].astype(str).eq(month).all(), f'{month}: assignment sai tháng'
    assert p.persona_id.notna().all() and p.persona_id.isin(PIDS).all()
    ids = pd.to_numeric(p.customer_id, errors='raise')
    assert ids.mod(1).eq(0).all()
    p['customer_id'] = ids.astype('int64')
    cols = ['customer_id','persona_id'] + (['persona_name'] if 'persona_name' in p else [])
    d = p[cols].merge(raw, on='customer_id', how='left', validate='one_to_one', indicator=True)
    matched = d['_merge'].eq('both')
    audit = dict(month=month, raw_snapshot_n=len(raw), assigned_n=len(p), matched_n=int(matched.sum()),
                 missing_raw_n=int((~matched).sum()), assignment_share_raw=len(p)/len(raw))
    if not matched.all(): raise ValueError(f'Assignment không có raw cùng tháng: {audit}')
    return d.drop(columns='_merge'), audit

joined = {}; audits = []
for m in MONTHS:
    p = pd.read_parquet(PERSONAS/f'personas_tree_{m}.parquet')
    joined[m], audit = join_personas(raw_months[m], p, m)
    audits.append(audit)
coverage = pd.DataFrame(audits)
coverage.to_csv(OUT/'join_coverage.csv', index=False)
display(coverage)
del raw_months

## 3. Quality và profile numeric
Báo nhóm P1–P6 và baseline `ALL_ASSIGNED`. Median/Q25/Q75 tính trên dữ liệu hợp lệ; không điền thu nhập giả cho vùng thiếu dữ liệu. `renta` là thu nhập hộ gia đình, không phải doanh thu ngân hàng. Các tỷ lệ không tự chứng minh nguyên nhân missing.

In [ ]:
def profile_month(d, month):
    quality=[]; numeric=[]; categorical=[]
    groups = [('ALL_ASSIGNED', d)] + list(d.groupby('persona_id', observed=True))
    for pid,g in groups:
        for c in RAW_COLS[2:]:
            miss = int(g[c+'_missing_raw'].sum())
            invalid = int(g[c+'_invalid'].sum()) if c in NUMERIC else 0
            quality.append(dict(month=month,persona_id=pid,feature=c,n=len(g),missing_raw_n=miss,
                invalid_n=invalid,valid_share=1-(miss+invalid)/len(g),missing_raw_share=miss/len(g)))
        for c in NUMERIC:
            v=g[c].dropna()
            numeric.append(dict(month=month,persona_id=pid,feature=c,n=len(g),valid_n=len(v),
                valid_share=len(v)/len(g),median=v.median(),q25=v.quantile(.25),q75=v.quantile(.75)))
        for c in CATEGORICAL:
            for cat,n in g[c].value_counts(dropna=False).items():
                categorical.append(dict(month=month,persona_id=pid,feature=c,category=str(cat),n=len(g),
                                        category_n=int(n),share=n/len(g)))
    return quality,numeric,categorical
qs=[]; ns=[]; cs=[]
for m,d in joined.items():
    q,n,c = profile_month(d,m); qs.extend(q); ns.extend(n); cs.extend(c)
quality=pd.DataFrame(qs); numeric=pd.DataFrame(ns); categorical=pd.DataFrame(cs)
base=numeric.loc[numeric.persona_id.eq('ALL_ASSIGNED'),['month','feature','median','q25','q75']].rename(
    columns={'median':'baseline_median','q25':'baseline_q25','q75':'baseline_q75'})
numeric=numeric.merge(base,on=['month','feature'],validate='many_to_one')
numeric['median_difference']=numeric['median']-numeric.baseline_median
numeric['median_diff_baseline_iqr']=numeric.median_difference/(numeric.baseline_q75-numeric.baseline_q25).replace(0,np.nan)
# Đây là chênh median chia IQR để mô tả, KHÔNG gọi là SMD hay p-value.
for name,table in [('quality_monthly',quality),('numeric_profiles',numeric),('categorical_profiles',categorical)]:
    table.to_csv(OUT/f'{name}.csv',index=False)
display(numeric.loc[numeric.month.eq(MONTHS[-1])].round(3))
display(quality.loc[quality.month.eq(MONTHS[-1])].round(3))

## 4. Category khác baseline và độ ổn định
So chênh lệch tỷ trọng theo **điểm phần trăm**. Missing được giữ là một category để audit nhưng không dùng làm đặc điểm nhân khẩu nổi bật. Không đặt tên mã kênh Kxx thành online/chi nhánh khi chưa có mapping.

Một đặc điểm được gắn `stable_candidate` nếu: đủ support/coverage, có dữ liệu ít nhất 3 tháng, nổi bật ít nhất 80% số tháng được đánh giá và không đổi hướng chênh lệch. Đây là screen mô tả có thể điều chỉnh; khách lặp qua tháng nên không coi các tháng độc lập. Baseline chứa cả persona đang so sánh.

In [ ]:
def categorical_comparison(categorical, quality):
    rows=[]
    for (month,feature),b in categorical.loc[categorical.persona_id.eq('ALL_ASSIGNED')].groupby(['month','feature']):
        baseline=b.set_index('category')['share']
        for pid in PIDS:
            z=categorical.loc[categorical.month.eq(month)&categorical.feature.eq(feature)&categorical.persona_id.eq(pid)]
            if z.empty: continue
            group_n=int(z.n.iloc[0]); counts=z.set_index('category').category_n
            qfeature={'age_group':'age','tenure_group':'antiguedad'}.get(feature,feature)
            qr=quality.loc[quality.month.eq(month)&quality.persona_id.eq(pid)&quality.feature.eq(qfeature)]
            valid_share=float(qr.valid_share.iloc[0])
            for cat,bs in baseline.items():
                cn=int(counts.get(cat,0)); share=cn/group_n; delta=share-bs
                reliable=group_n>=MIN_GROUP_N and valid_share>=MIN_VALID_SHARE
                notable=reliable and cat!=MISSING and cn>=MIN_CATEGORY_N and abs(delta)>=MIN_ABS_SHARE_DIFF
                rows.append(dict(month=month,persona_id=pid,feature=feature,category=cat,n=group_n,category_n=cn,
                    share=share,baseline_share=bs,difference_pp=100*delta,valid_share=valid_share,
                    reliable=reliable,notable=notable))
    return pd.DataFrame(rows)
comparison=categorical_comparison(categorical,quality)
rows=[]
for (pid,f,cat),g in comparison.groupby(['persona_id','feature','category']):
    good=g.loc[g.reliable]; notable=good.loc[good.notable]
    same_direction=len(notable)>0 and (notable.difference_pp.gt(0).all() or notable.difference_pp.lt(0).all())
    # Mẫu số gồm mọi tháng persona có xuất hiện, không chỉ những tháng có support tốt.
    fraction=len(notable)/len(g)
    rows.append(dict(persona_id=pid,feature=f,category=cat,months=len(g),reliable_months=len(good),
        notable_months=len(notable),notable_fraction=fraction,
        difference_pp_min=g.difference_pp.min(),difference_pp_max=g.difference_pp.max(),
        stable_candidate=len(g)>=MIN_STABLE_MONTHS and fraction>=MIN_STABLE_FRACTION and same_direction))
stability=pd.DataFrame(rows)
numeric_stability=numeric.loc[~numeric.persona_id.eq('ALL_ASSIGNED')].groupby(['persona_id','feature']).agg(
    months=('month','nunique'),median_min=('median','min'),median_max=('median','max'),
    coverage_min=('valid_share','min'),difference_min=('median_difference','min'),difference_max=('median_difference','max')).reset_index()
comparison.to_csv(OUT/'category_differences.csv',index=False)
stability.to_csv(OUT/'category_stability.csv',index=False)
numeric_stability.to_csv(OUT/'numeric_stability.csv',index=False)
display(stability.loc[stability.stable_candidate].round(3))
display(numeric_stability.round(3))

## 5. Bảng profile tháng cuối và thẻ demographics
Thẻ chỉ trình bày số quan sát và ứng viên đặc điểm. Chưa tự suy ra hành động bán sản phẩm từ tuổi/thu nhập; kết nối CLV/affinity đã làm ở notebook 12/13. Nhóm không khác demographics vẫn là persona hành vi hợp lệ.

In [ ]:
latest=MONTHS[-1]; cards=[]; wide=[]
for pid,g in joined[latest].groupby('persona_id',observed=True):
    name=str(g.persona_name.iloc[0]) if 'persona_name' in g else pid
    lines=[f'## {pid} — {name}',f'- Tháng {latest}: {len(g):,} khách; {len(g)/len(joined[latest]):.1%} cohort được gán persona.']
    row=dict(month=latest,persona_id=pid,persona_name=name,n=len(g),share_assigned=len(g)/len(joined[latest]))
    for f in NUMERIC:
        r=numeric.loc[numeric.month.eq(latest)&numeric.persona_id.eq(pid)&numeric.feature.eq(f)].iloc[0]
        value=f'{r["median"]:,.1f} (Q25–Q75: {r.q25:,.1f} - {r.q75:,.1f})' if r.valid_n else 'Không có dữ liệu hợp lệ'
        row[f]=value;row[f+'_coverage']=r.valid_share
        lines.append(f'- {f}: {value}; coverage {r.valid_share:.1%}; median baseline {r.baseline_median:,.1f}.')
    for f in ['canal_entrada','segmento','sexo','nomprov']:
        z=categorical.loc[categorical.month.eq(latest)&categorical.persona_id.eq(pid)&categorical.feature.eq(f)&categorical.category.ne(MISSING)].nlargest(3,'share')
        label='; '.join(f'{r.category}: {r.share:.1%}' for r in z.itertuples()) or 'Không có dữ liệu'
        row[f]=label; lines.append(f'- Top {f}: {label}.')
    candidates=stability.loc[stability.persona_id.eq(pid)&stability.stable_candidate]
    # Chỉ đưa lên card nếu tháng cuối vẫn đủ điều kiện nổi bật.
    current_notable=comparison.loc[comparison.month.eq(latest)&comparison.persona_id.eq(pid)&comparison.notable]
    candidates=candidates.merge(current_notable[['feature','category']],on=['feature','category'],how='inner')
    lines.append('- Đặc điểm ổn định cần diễn giải: '+('; '.join(f'{r.feature}={r.category}, chênh {r.difference_pp_min:.1f} - {r.difference_pp_max:.1f} điểm % qua tháng' for r in candidates.itertuples()) or 'Chưa có đặc điểm vượt screen; giữ mô tả bằng số liệu.'))
    cards.append('\n'.join(lines));wide.append(row)
profile_latest=pd.DataFrame(wide)
profile_latest.to_csv(OUT/'persona_demographics_latest.csv',index=False)
(OUT/'persona_demographics_cards.md').write_text('# Demographics P1–P6\n\n'+'\n\n'.join(cards),encoding='utf-8')
config=dict(version='persona-demographics-v1',raw_path=str(RAW),persona_dir=str(PERSONAS),months=MONTHS,
    baseline='ALL_ASSIGNED within each month',imputation=False,age_max_screen=AGE_MAX,
    screen=dict(min_group_n=MIN_GROUP_N,min_valid_share=MIN_VALID_SHARE,min_abs_share_diff=MIN_ABS_SHARE_DIFF,
                min_category_n=MIN_CATEGORY_N,min_stable_months=MIN_STABLE_MONTHS,min_stable_fraction=MIN_STABLE_FRACTION),
    limitations=['descriptive association, not causation','repeated customers across months','no acquisition evaluation',
                 'no inferred channel mapping','income is household income, not bank revenue'])
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
display(profile_latest)
print('Đã xuất:',OUT.resolve())

## Checklist đọc kết quả
- `join_coverage.csv`: assignment có khớp raw đúng tháng không, cohort bao phủ bao nhiêu snapshot?
- `quality_monthly.csv`: đặc biệt xem renta missing theo persona. Không diễn giải median có coverage thấp.
- `numeric_profiles.csv`: so median và IQR; tương quan tuyến tính thấp không có nghĩa biến vô dụng.
- `category_differences.csv`: đọc cả tỷ trọng nhóm và baseline, không chỉ top category.
- `category_stability.csv`: stable_candidate là ứng viên diễn giải, không phải kết quả kiểm định.
- `persona_demographics_cards.md`: ghép vào persona cards chính sau khi xem bảng, không thay đổi rules P1–P6.
- Nếu cần thu nhập thấp/vừa/cao: học ngưỡng ở tháng tham chiếu rồi khóa; notebook hiện giữ median/IQR để tránh tạo ngưỡng tùy tiện.

EDA nền cũ nên sửa riêng: Added phải có cặp tháng liên tiếp; so cohort bằng snapshot first-seen; không gộp tuổi trẻ em thành 18; phân biệt mất quan sát với churn thật. Notebook này đọc raw riêng nên không kế thừa phép sửa tuổi trong EDA cũ.